# Nieoznakowane radiowozy

Generated for a selected file in Catalog Explorer. This notebook shows how to do the following: - viewing file metadata - parsing documents with ai_parse_document - analyzing images with ai_query

# Installs and imports

In [0]:
%pip install osmium --quiet
%pip install folium --quiet

In [0]:
import re
from pyspark.sql.functions import length, explode, split, lit, expr, col, explode_outer, split, regexp_replace, row_number, element_at
from pyspark.sql.window import Window
import osmium
import folium
import itertools
from collections import defaultdict

# Parameters

In [0]:
MAP_PATH = "/Volumes/workspace/default/undercover_cars/maps/poland-261101.osm.pbf"
FILE_PATH = "/Volumes/workspace/default/undercover_cars/nieoznakowane-radiowozy-posts.csv"
IMAGES_VOLUME_DIR = "/Volumes/workspace/default/undercover_cars/images"

# Loading feed with list of Facebook posts regarding undercover police cars

In [0]:
df = spark.read.csv(FILE_PATH, header=True, multiLine=True, escape='"', inferSchema=True)

In [0]:

# Split the space-separated Tags string into an array of individual tags, then replace the original Tags column with the new array column.
df = df.withColumn("TagsStruct", expr("split(trim(Tags), ' ')"),)
df = df.drop("Tags").withColumnRenamed("TagsStruct", "Tags")
display(df)

In [0]:
# Drop posts whose "Post text" contains the phrase "ROBITY RADIOWÓZ"
# (e.g. reports about crashed police cars, not undercover ones).
# Applied on the raw text; equivalent to checking any item of the
# later-split Post text array for the phrase.
df = df.filter(~col("`Post text`").contains("ROBITY RADIOWÓZ"))

display(df)

# Read related mappings

In [0]:
dim_pl_registration_plates_pattern = spark.read.table("workspace.undercover_cars.dim_pl_registration_plates_pattern")

### # 1st Plate discovery method - Extract by pattern from Post tags

In [0]:
# Known prefixes from the dimension table, unioned with the ground-truth series.
dim_prefixes = {
    row["plate_prefix"]
    for row in dim_pl_registration_plates_pattern.select("plate_prefix").collect()
}

pl_plate_prefixes = sorted(dim_prefixes, key=len, reverse=True)
prefixes_regex = "|".join(pl_plate_prefixes)
PLATE_PATTERN = f"({prefixes_regex})([A-Z0-9]{{4,5}})(?![A-Z0-9])"

df = (
    df.filter(df["PostClass"] == "1-UndercoverCar")
    .withColumn("Post text",
        expr("filter(split(`Post text`, '(?i)<br\\s*/?>|\\n'), p -> trim(p) != '')"),)
    .withColumn(
        "plates",
        expr(
            "regexp_extract_all(concat_ws(' ', `Tags`), '"
            + PLATE_PATTERN.replace("\\", "\\\\")
            + "', 0)"
        ),
    )
    # Mark the discovery source as "1-tags" only when the plates array from Tags
    # is non-null and non-empty; otherwise leave plate_discovery_source NULL.
    .withColumn("plate_discovery_source", expr("case when size(plates) > 0 then '1-tags' end"))
)

display(df)

### # 2nd Plate discovery method - Extract by pattern from Post text

In [0]:
# get the tag that its lengths is between 5 and 10 characters, trim unprinted characters and spaces, check if they match plate pattern and return it in "plates_candidate" column

df_temp = df.filter(df["plates"].isNull() | (expr("size(plates) = 0"))) \
.withColumn(
    "plates",
    expr("""
        filter(
            transform(`Post text`, p -> trim(regexp_replace(p, '[^\\x20-\\x7E]', ''))),
            p -> length(p) >= 5 and length(p) <= 10
        )
    """),) \
    .withColumn("plate_discovery_source", expr("case when size(plates) > 0 then '2-post-text' end"))

# Merge df_temp records by "Post ID" into df, overwriting matching rows
# (rows in df_temp win), then delete df_temp. df_temp has the same columns as
# df (filter + withColumn), so unionByName is safe.
df = (
    df
    .join(df_temp.select("Post ID"), "Post ID", "left_anti")  # drop rows overwritten by df_temp
    .unionByName(df_temp)
)

del df_temp

display(df)

### # 3rd Plate discovery method - OCR Regocnition with AI

In [0]:
df_investigated = df.filter(
    df["plates"].isNull() | (expr("size(plates) = 0"))
)

In [0]:
# Add image_base64: base64-encoded binary content of the post's image file from the Volume (null when no file was saved for that Post ID).



df_images = (
    spark.read.format("binaryFile")
    .load(IMAGES_VOLUME_DIR)
    .withColumn(
        "Post ID",
        regexp_replace(element_at(split(col("path"), "/"), -1), r"\.[^.]+$", ""),
    )
    .select("Post ID", "content")
)

df_investigated = (
    df.filter(
    df["plates"].isNull() | (expr("size(plates) = 0"))
)
    .join(df_images, "Post ID", "left")
    .withColumn("image_base64", expr("base64(content)"))
    .drop("content")
)

In [0]:
# Use ai_query with a vision-capable model to read the registration plate from each image.
PLATE_PROMPT = (
    "Extract the vehicle registration plate text from this image. "
    "Return only the plate number in uppercase with no spaces or separators, "
    "or an empty string if no plate is visible or it is unreadable."
)

# df_investigated has image_base64 (the raw binary content column was dropped earlier),so decode it back to BINARY for the files => argument.
df_investigated = (
    df_investigated
    .withColumn("_plate_ai", expr(f"""
        ai_query(
            'system.ai.llama-4-maverick',
            '{PLATE_PROMPT}',
            files => unbase64(image_base64),
            failOnError => false
        )
        """))
        .withColumn("plates", expr("""
        case
            when _plate_ai.errorMessage is not null then cast(null as array<string>)
            when nullif(upper(trim(_plate_ai.result)), '') is null then cast(null as array<string>)
            else array(upper(trim(_plate_ai.result)))
        end
        """))
    .withColumn("plate_discovery_source", expr("case when plates is not null then '3-ai-recognition' end"))
    .drop("_plate_ai", "image_base64")
)


In [0]:
df = (
    df
    .join(df_investigated.select("Post ID"), "Post ID", "left_anti")  # drop rows overwritten by df_investigated
    .unionByName(df_investigated)
)

In [0]:
unresolved_count = df.filter(df["plate_discovery_source"].isNull()).count()
assert unresolved_count == 0, "Not all posts and registration plates recognition processes went correctly."


In [0]:
match_rank = Window.partitionBy("Post ID", "plate").orderBy(length("plate_prefix").desc())

df = (
    df
    .withColumn("plate", explode_outer("plates"))
    .join(
        dim_pl_registration_plates_pattern.select("plate_prefix", "Entity"),
        expr("plate LIKE concat(plate_prefix, '%')"),
        "left",
    )
    .withColumn("_prefix_rank", row_number().over(match_rank))
    .filter(col("_prefix_rank") == 1)
    .drop("plate_prefix", "_prefix_rank")
)

display(df)




# To-do:
1. ~~Update MD file feed (with ImageURL)~~
2. For df_investigated run analysis of image to search for text (that matches registration plates pattern)
3. ~~Geofabric Openstreetmap (upload map of Poland to Volume)~~
4. Apache Sedona (https://sedona.apache.org/latest/sedonaspark/#__tabbed_1_2)
5. folium 
6. Update (on-regular basis) file on the Volume https://download.geofabrik.de/europe/poland.html


In [0]:
exit(0)

In [0]:


HIGHWAY_REF = "A4"  # primary road number to extract


class HighwayWayHandler(osmium.SimpleHandler):
    """Collect geometry of motorway ways whose ref includes HIGHWAY_REF."""

    def __init__(self, ref):
        super().__init__()
        self.ref = ref
        self.segments = []

    def way(self, w):
        if w.tags.get("highway") != "motorway":
            return
        refs = w.tags.get("ref")
        if not refs or self.ref not in refs.split(";"):
            return
        try:
            coords = [(n.lat, n.lon) for n in w.nodes]
        except osmium.InvalidLocationError:
            return  # way with an unresolvable node — skip it
        if len(coords) >= 2:
            self.segments.append(coords)


handler = HighwayWayHandler(HIGHWAY_REF)
# locations=True builds a node location index in C++ while scanning the file,
# so w.nodes above yields resolved (lat, lon) coordinates for every way.
handler.apply_file(MAP_PATH, locations=True)

segments = handler.segments
if not segments:
    raise ValueError(
        f"No {HIGHWAY_REF} motorway geometry found in {MAP_PATH} "
        "— check the file and the highway ref."
    )

all_points = [pt for seg in segments for pt in seg]
center = [
    sum(lat for lat, _ in all_points) / len(all_points),
    sum(lon for _, lon in all_points) / len(all_points),
]
print(f"Read {len(segments)} A4 segments ({len(all_points)} points) from the local PBF. Center: {center}")
print(
    "Bounds:",
    [min(lat for lat, _ in all_points), min(lon for _, lon in all_points)],
    [max(lat for lat, _ in all_points), max(lon for _, lon in all_points)],
)

m = folium.Map(location=center, zoom_start=7, tiles="OpenStreetMap")

for seg in segments:
    folium.PolyLine(
        seg,
        color="#1a73e8",
        weight=5,
        opacity=0.8,
        tooltip="A4 – autostrada",
    ).add_to(m)

# Fit the map bounds to the full extent of the A4.
m.fit_bounds(
    [
        [min(lat for lat, _ in all_points), min(lon for _, lon in all_points)],
        [max(lat for lat, _ in all_points), max(lon for _, lon in all_points)],
    ]
)

# Render the folium HTML inline in the notebook.
displayHTML(m._repr_html_())


In [0]:
# Reusable function version of the A4 map cell: draw an interactive OSM (folium) map
# for a set of roads, each annotated with the geo-districts it is associated with.
#
# Input: a list of dictionaries, each with ONE entry:
#   KEY   -> road mark (e.g. "A4", "S5", "DK94", "DW345" — as used in the OSM `ref` tag)
#   VALUE -> geo-districts for that road: a single string or a list of strings
#            (e.g. "Województwo Dolnośląskie" or ["Województwo Dolnośląskie", "Powiat Wrocław"])
#
# The districts are attached to the road as tooltip labels (the local PBF has no
# administrative boundaries, so districts cannot be used to clip geometry).


MAP_PATH = "/Volumes/workspace/default/undercover_cars/maps/poland-261101.osm.pbf"

ROAD_COLORS = ["#1a73e8", "#d93025", "#188038", "#f29900", "#9334e6", "#00897b"]


class RoadWayHandler(osmium.SimpleHandler):
    """Collect geometry of all ways whose ref tag contains one of the wanted road marks."""

    def __init__(self, road_marks):
        super().__init__()
        self.road_marks = set(road_marks)
        self.segments = defaultdict(list)  # road mark -> list of [(lat, lon), ...]

    def way(self, w):
        refs = w.tags.get("ref")
        if not refs:
            return
        for ref in refs.split(";"):
            ref = ref.strip()
            if ref in self.road_marks:
                try:
                    coords = [(n.lat, n.lon) for n in w.nodes]
                except osmium.InvalidLocationError:
                    return  # way with an unresolvable node — skip it
                if len(coords) >= 2:
                    self.segments[ref].append(coords)


def render_roads_map(road_districts, map_path=MAP_PATH, m=None, render_html=True):
    """Render roads on an interactive OpenStreetMap (folium) map.

    Args:
        road_districts: list of dicts; each dict maps one road mark (KEY, e.g. "A4",
            "DK94") to its geo-districts (VALUE: a string or list of strings).
        map_path: path to the Geofabrik Poland OSM extract (PBF) on the Volume.
        m: optional existing folium.Map to add the roads to (a new map is created otherwise).
        render_html: when True, display the map inline in the notebook.

    Returns:
        The folium.Map object with one colored PolyLine per road, labelled with its districts.
    """
    # Normalize input: {road mark -> [district, ...]} (later entries extend earlier ones).
    roads = {}
    for entry in road_districts:
        for road_mark, districts in entry.items():
            district_list = [districts] if isinstance(districts, str) else list(districts)
            roads.setdefault(road_mark, []).extend(district_list)

    handler = RoadWayHandler(roads.keys())
    # locations=True builds a node location index in C++ while scanning the file,
    # so w.nodes yields resolved (lat, lon) coordinates for every way.
    handler.apply_file(map_path, locations=True, timeout=300)

    missing = sorted(set(roads) - set(handler.segments))
    if missing:
        print(f"WARNING: no geometry found for road mark(s): {', '.join(missing)}")

    all_points = [
        pt
        for segments in handler.segments.values()
        for segment in segments
        for pt in segment
    ]
    if not all_points:
        raise ValueError(
            f"None of the requested roads found in {map_path} — check the road marks and the file."
        )

    if m is None:
        center = [
            sum(lat for lat, _ in all_points) / len(all_points),
            sum(lon for _, lon in all_points) / len(all_points),
        ]
        m = folium.Map(location=center, zoom_start=7, tiles="OpenStreetMap")

    color_cycle = itertools.cycle(ROAD_COLORS)
    for road_mark, segments in handler.segments.items():
        districts = ", ".join(dict.fromkeys(roads[road_mark]))  # dedupe, keep order
        color = next(color_cycle)
        for segment in segments:
            folium.PolyLine(
                segment,
                color=color,
                weight=5,
                opacity=0.8,
                tooltip=f"{road_mark} – {districts}",
            ).add_to(m)

    # Fit the map bounds to the full extent of all drawn roads.
    m.fit_bounds(
        [
            [min(lat for lat, _ in all_points), min(lon for _, lon in all_points)],
            [max(lat for lat, _ in all_points), max(lon for _, lon in all_points)],
        ]
    )

    if render_html:
        displayHTML(m._repr_html_())

    return m


# Example usage:
road_districts = [
    {"A4": ["Województwo Dolnośląskie", "Województwo Opolskie"]},
    {"DK94": ["Województwo Dolnośląskie", "Powiat Wrocław"]},
]

render_roads_map(road_districts)
